# MEG Faces Main Analysis

This is a companion tutorial to the paper *"A Primer on Low-Dimensional Neural Dynamics: PCA-Based Trajectory Analysis for EEG and MEG."*

Here we apply the **10-step PCA trajectory workflow** to the **Wakeman–Henson multimodal face-processing dataset**, asking how the MEG response evolves for **Famous faces**, **Unfamiliar faces** and **Scrambled images**. All three conditions are placed in one shared PCA space, and two planned comparisons are then evaluated *without refitting the axes*:

1. **Faces vs Scrambled** — does face structure alter the trajectory?
2. **Famous vs Unfamiliar** — does familiarity alter the trajectory?

The dataset is chosen deliberately. Face processing has one of the best-characterised evoked signatures in cognitive neuroscience — the N170 — so if a trajectory analysis that is never told faces exist recovers a divergence at roughly the right latency, that is a check on the method rather than a discovery about faces. Familiarity, by contrast, is a much later and smaller effect, which makes the pair a useful test of what the geometry can and cannot see.

<div class="alert alert-secondary">
<b>🗺️ Position in the tutorial series:</b><br>
<ol style="margin-bottom: 0; margin-top: 5px;">
  <li><b>EEGBCI Introductory Tutorial:</b> the core 10-step PCA workflow on 64-channel EEG.</li>
  <li><b>EEGBCI Nonlinear Tutorial:</b> PCA against UMAP, PHATE and Isomap, with geometry diagnostics.</li>
  <li><b>EEGBCI Decoding Tutorial:</b> trajectories for cross-participant BCI decoding.</li>
  <li><b>MEG Faces Main Analysis (this notebook):</b> the same workflow on 306-sensor whitened MEG.</li>
  <li><b>MEG Spectral Envelopes:</b> the same workflow on band-limited amplitude rather than the broadband ERF.</li>
  <li><b>MEG Faces Decoding:</b> single-trial cross-participant prediction and temporal-shape alignment.</li>
</ol>
</div>

### What changes when you move from EEG to MEG

Three things, all of which the workflow has to accommodate:

- **Two sensor types with different units.** Magnetometers measure field (T), gradiometers measure field gradient (T/m). Their raw numbers are not comparable, so a variance-based method would silently be dominated by whichever has the larger numerical scale. The preprocessing whitens each participant with their own **empty-room covariance**, which normalizes by measured noise instead of an arbitrary display constant.
- **Far more sensors, and higher effective dimensionality.** With 306 channels the variance is genuinely more spread out than in 64-channel EEG, so PC1 carries a smaller share and more components are needed to describe the same fraction of the response.
- **Head position matters.** Maxwell filtering with movement compensation places every participant's data into a common device-independent frame — the closest thing MEG has to a shared anatomical coordinate system.

### Dataset & expected outputs

- **Dataset:** Wakeman–Henson `ds000117`, prepared MEG derivatives. The analysis is **sensor-space and descriptive**; it is not a source-localisation analysis.
- **Conditions:** Famous, Unfamiliar, Scrambled.
- **Outputs:** the figures and tables shown inline, plus an optional compact export in Step 12. The headless companion `scripts/analysis_megfaces_main.py` writes the complete bundle including a self-contained HTML report.

## 0. Setup & Configuration

Every scientific choice is declared before any data is loaded. Downloading and preprocessing are deliberately kept outside this notebook: `_load_wakeman_henson_container` reads derivatives that have already been Maxwell-filtered, epoch-rejected with per-participant thresholds, and whitened.

<div class="alert alert-info">
<b>⚙️ Analysis choices:</b><br>
<ul style="margin-bottom: 0;">
<li><code>SUBJECTS</code> — participants to analyze. Only those actually present in <code>DERIVATIVES_ROOT</code> are loaded, so the list can be left at the full cohort.</li>
<li><code>SENSOR_SET</code> — <code>"all_sensors"</code>, <code>"sensors_occipital"</code>, <code>"sensors_temporal"</code>, or <code>"sensors_occipito_temporal"</code>. These are <b>VectorView helmet selections, not source-localized cortical ROIs</b>: an "occipital" result is a statement about where the field was strongest outside the head.</li>
<li><code>METRIC_PCA_MODE</code> — <code>"shared"</code>, <code>"subject"</code>, or <code>"both"</code>, controlling which coordinate system the participant-level metrics are computed in. Step 5a explains why this switch exists.</li>
</ul>
</div>

In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

from coco_pipe.dim_reduction import DimReduction
from coco_pipe.viz.interactive import (
    plot_scree,
    plot_timecourses,
    plot_trajectory,
    plot_trajectory_metric_series,
)
from pca_neural_trajectories import facet_figures, overlay_figures
from pca_neural_trajectories.wakeman_henson import (
    LABEL_NAMES,
    MEG_SENSOR_SETS,
    _load_wakeman_henson_container,
)

SEED = 42
DERIVATIVES_ROOT = Path(
    os.getenv(
        "MEG_DERIVATIVES_ROOT",
        str(Path.home() / "mne_data" / "ds000117" / "derivatives" / "pca_trajectories"),
    )
)
# Every prepared participant; the loader silently skips ids that are absent,
# so this can stay at the full cohort on a machine holding only a subset.
SUBJECTS = tuple(f"{subject:02d}" for subject in range(1, 4))
SENSOR_SET = "all_sensors"

# Controls the coordinate system used for participant-level metrics.
# Choose "shared", "subject", or "both".
METRIC_PCA_MODE = "both"

N_COMPONENTS = 10
N_DISPLAY_COMPONENTS = 3
ACTIVE_WINDOW = (0.0, 0.6)

CONDITION_COLORS = {1: "#0072B2", 2: "#D55E00", 3: "#009E73"}
CONDITION_FILLS = {
    1: "rgba(0, 114, 178, 0.15)",
    2: "rgba(213, 94, 0, 0.15)",
    3: "rgba(0, 158, 115, 0.15)",
}
CONTRAST_COLORS = {
    "Faces vs Scrambled": "#7B2CBF",
    "Famous vs Unfamiliar": "#D55E00",
}
CONTRAST_FILLS = {
    "Faces vs Scrambled": "rgba(123, 44, 191, 0.15)",
    "Famous vs Unfamiliar": "rgba(213, 94, 0, 0.15)",
}

if METRIC_PCA_MODE not in {"shared", "subject", "both"}:
    raise ValueError('METRIC_PCA_MODE must be "shared", "subject", or "both".')
if SENSOR_SET not in MEG_SENSOR_SETS:
    raise ValueError(f"SENSOR_SET must be one of {tuple(MEG_SENSOR_SETS)}.")

pio.templates.default = "plotly_white"
rng = np.random.default_rng(SEED)

## Step 1. Choose the Representation (State Space)

Before computing any trajectory we must define the space it lives in. At each time point, the MEG state is the pattern across the selected sensors:

$$\mathbf{x}(t) \in \mathbb{R}^{N_{\mathrm{sensors}}}.$$

<div class="alert alert-info">
<b>🧠 The conceptual shift:</b><br>
Instead of 306 squiggly lines evolving in parallel (the traditional ERF view), think of the brain as a <b>single point</b> moving through a 306-dimensional room. The path that point traces as time advances is the <b>neural trajectory</b>, and everything in this notebook is a property of that path: where it goes, how fast, how far apart two conditions' paths get, and when.
</div>

The preprocessing has already whitened each participant with that participant's empty-room covariance. That places magnetometers and gradiometers on a common noise scale before PCA sees them, which is why we **do not z-score the channels again** — a second normalization would throw away the measured noise scaling and replace it with an arbitrary one.

<div class="alert alert-warning">
<b>⚠️ Interpretation boundary:</b><br>
Sensor-space PCs are mixtures of sources that were already mixed by volume conduction and the forward field. We interpret trajectory <i>geometry and timing</i>; we do not read a PC as a single anatomical generator.
</div>

## Step 2. Preprocess & Load the Prepared Participants

The loader returns a tensor of shape `trial × sensor × time`, with labels attached at the trial level (including repetition metadata for later auditing).

The preparation that produced these derivatives did the work that PCA cannot do for itself:

1. <span style="color: #D55E00;"><b>Maxwell filtering (SSS) with movement compensation</b></span>, using an explicit head-coordinate sphere origin rather than an automatic fit — an ill-conditioned SSS basis distorts the data silently rather than loudly.
2. <span style="color: #D55E00;"><b>Per-participant epoch rejection thresholds</b></span> derived by `autoreject`. A single fixed peak-to-peak threshold rejects participants for being *loud* rather than for being *bad*, and can discard most of a perfectly good recording.
3. <span style="color: #D55E00;"><b>Empty-room whitening</b></span> with the data rank estimated from the data rather than trusted from the SSS record.

<div class="alert alert-warning">
<b>🧠 Why trial counts differ between participants:</b><br>
Because rejection thresholds are per-participant, retained trial counts vary. This matters twice below: participants with more retained trials contribute slightly more to the pooled PCA basis (Step 4), and so every <i>group</i> summary is computed by averaging within participant first, giving each participant equal weight regardless of trial count.
</div>

In [2]:
available_subjects = tuple(
    path.name.removeprefix("sub-")
    for path in sorted(DERIVATIVES_ROOT.glob("sub-*"))
    if path.is_dir()
)
print(f"Prepared locally: {', '.join(available_subjects)}")
print(f"Selected now    : {', '.join(SUBJECTS)}")

container = _load_wakeman_henson_container(
    DERIVATIVES_ROOT,
    subjects=SUBJECTS,
    sensor_set=SENSOR_SET,
)

X = np.asarray(container.X, dtype=np.float32)
times = np.asarray(container.coords["time"], dtype=float)
channels = np.asarray(container.coords["channel"]).astype(str)
subjects = np.asarray(container.coords["subject"]).astype(str)
labels = np.asarray(container.y, dtype=int)
repetitions = np.asarray(container.coords["repetition"]).astype(str)

unique_subjects = np.unique(subjects)
conditions = np.array(sorted(LABEL_NAMES))

print(f"data shape : {X.shape}  (trials, sensors, time)")
print(f"time window: {times[0]:+.3f} to {times[-1]:+.3f} s")
print(f"sampling   : {1 / np.diff(times).mean():.1f} Hz")
print(f"subjects   : {', '.join(unique_subjects)}")
print(f"sensor set : {SENSOR_SET} ({len(channels)} sensors)")
print(f"whitening  : {container.meta.get('whitening', 'not recorded')}")

Prepared locally: 01, 02, 03, 04, 05, 06
Selected now    : 01, 02, 03
Reading /Users/hamzaabdelhedi/mne_data/ds000117/derivatives/pca_trajectories/sub-01/ses-meg/meg/sub-01_ses-meg_task-facerecognition_epo.fif ...
    Found the data of interest:
        t =    -200.00 ...     800.00 ms
        0 CTF compensation matrices available
Adding metadata with 6 columns
886 matching events found
No baseline correction applied
0 projection items activated


/Users/hamzaabdelhedi/Projects/research/pca-neural-trajectories-eeg-meg/pca_neural_trajectories/wakeman_henson.py:660: RuntimeWarning: Something went wrong in the data-driven estimation of the data rank as it exceeds the theoretical rank from the info (74 > 72). Consider setting rank to "auto" or setting it explicitly as an integer.
  whitener, ch_names = mne.cov.compute_whitener(


Reading /Users/hamzaabdelhedi/mne_data/ds000117/derivatives/pca_trajectories/sub-02/ses-meg/meg/sub-02_ses-meg_task-facerecognition_epo.fif ...
    Found the data of interest:
        t =    -200.00 ...     800.00 ms
        0 CTF compensation matrices available
Adding metadata with 6 columns
881 matching events found
No baseline correction applied
0 projection items activated


/Users/hamzaabdelhedi/Projects/research/pca-neural-trajectories-eeg-meg/pca_neural_trajectories/wakeman_henson.py:660: RuntimeWarning: Something went wrong in the data-driven estimation of the data rank as it exceeds the theoretical rank from the info (73 > 72). Consider setting rank to "auto" or setting it explicitly as an integer.
  whitener, ch_names = mne.cov.compute_whitener(


Reading /Users/hamzaabdelhedi/mne_data/ds000117/derivatives/pca_trajectories/sub-03/ses-meg/meg/sub-03_ses-meg_task-facerecognition_epo.fif ...
    Found the data of interest:
        t =    -200.00 ...     800.00 ms
        0 CTF compensation matrices available
Adding metadata with 6 columns
585 matching events found
No baseline correction applied
0 projection items activated


/Users/hamzaabdelhedi/Projects/research/pca-neural-trajectories-eeg-meg/pca_neural_trajectories/wakeman_henson.py:660: RuntimeWarning: Something went wrong in the data-driven estimation of the data rank as it exceeds the theoretical rank from the info (72 > 71). Consider setting rank to "auto" or setting it explicitly as an integer.
  whitener, ch_names = mne.cov.compute_whitener(


data shape : (2352, 306, 251)  (trials, sensors, time)
time window: -0.200 to +0.800 s
sampling   : 250.0 Hz
subjects   : 01, 02, 03
sensor set : all_sensors (306 sensors)
whitening  : subject-wise empty-room covariance


In [3]:
condition_names = pd.Series([LABEL_NAMES[c] for c in labels], name="condition")
trial_counts = pd.crosstab(
    pd.Series(subjects, name="subject"),
    condition_names,
).reindex(columns=[LABEL_NAMES[c] for c in conditions])

print("Retained trials per participant and condition")
display(trial_counts)

print("Repetition counts")
display(pd.crosstab(condition_names, repetitions))

Retained trials per participant and condition


condition,Famous,Unfamiliar,Scrambled
subject,,,
01,294,295,297
02,293,296,292
03,206,196,183


Repetition counts


col_0,delayed,first,immediate
condition,,,
Famous,181,405,207
Scrambled,199,393,180
Unfamiliar,206,396,185


### Sensor-space sanity check

Before reducing anything, verify that stimulus-locked structure is actually visible in the whitened sensor data. For each participant and condition we average trials and compute the root-mean-square response across sensors.

This is a cheap but non-negotiable step: PCA will happily return components from data containing nothing but noise, and a flat RMS here would mean every geometric result downstream is describing noise geometry. It is a compact quality-control view, not a replacement for inspecting the original epochs.

In [4]:
sensor_magnitude = {c: [] for c in conditions}

for subject in unique_subjects:
    for condition in conditions:
        rows = (subjects == subject) & (labels == condition)
        evoked = X[rows].mean(axis=0)
        sensor_magnitude[condition].append(np.sqrt(np.mean(evoked**2, axis=0)))

# One curve per participant, grouped by condition: plot_timecourses draws the
# group mean with an SEM band.
sensor_curves = np.concatenate(
    [np.asarray(sensor_magnitude[condition]) for condition in conditions]
)[:, np.newaxis, :]
sensor_groups = [
    LABEL_NAMES[condition]
    for condition in conditions
    for _ in sensor_magnitude[condition]
]

fig_sensor = plot_timecourses(
    sensor_curves,
    times=times,
    channel_names=["RMS"],
    group_labels=sensor_groups,
    palette={LABEL_NAMES[c]: CONDITION_COLORS[c] for c in conditions},
    error_style="band",
    xlabel="Time (s)",
    ylabel="RMS response (noise-normalised units)",
    title="Whitened sensor-space response magnitude",
)
fig_sensor.add_vline(x=0, line_dash="dash", line_color="black")
fig_sensor.show()


## Step 3. Reshape into (Samples × Features)

PCA expects `observations × features`. One observation is one time point from one trial, and the features are MEG sensors:

$$(\text{trials},\ \text{sensors},\ \text{time}) \rightarrow
(\text{trials} \times \text{time},\ \text{sensors}).$$

<div class="alert alert-info">
<b>📦 Two deliberate choices in this reshape:</b><br>
<ul style="margin-bottom: 0;">
<li><b>Single trials, not averages.</b> Every time point of every trial is an independent observation. Averaging first would give PCA only the evoked response and hide trial-to-trial variance that is part of the population geometry.</li>
<li><b>Labels are withheld.</b> The fit never sees a condition, so it cannot manufacture a difference between conditions. Any separation found in Step 7 was already present in the variance structure — which is exactly what makes the N170 falling out of the analysis a meaningful check.</li>
</ul>
</div>

Because all retained observations enter the fit, participants with more retained trials weigh slightly more in the basis; the count table above makes that weighting visible.

In [5]:
n_trials, n_sensors, n_times = X.shape
pooled = X.transpose(0, 2, 1).reshape(n_trials * n_times, n_sensors)

print(f"input tensor : {X.shape}")
print(f"PCA matrix   : {pooled.shape}  (trial-time observations, sensors)")

input tensor : (2352, 306, 251)
PCA matrix   : (590352, 306)  (trial-time observations, sensors)


## Step 4. Fit the PCA Model

A single PCA basis is fitted across all participants and conditions, so every trajectory and contrast below lives in the same coordinate system. Without that, "the distance between two conditions" would mean something different for each participant and could not be averaged.

We fit ten components so the dimensionality can be inspected, and use the first three for the main figures.

<div class="alert alert-info">
<b>💡 Three PCs are a display choice, not a claim:</b><br>
The scree plot and cumulative variance in Step 5 show how much structure the 3-D view omits — and for 306-channel MEG that omission is larger than for 64-channel EEG, because the variance is spread across more directions. A clean 3-D figure is not evidence that the MEG response is intrinsically three-dimensional.
</div>

In [6]:
shared_pca = DimReduction(
    method="PCA",
    n_components=N_COMPONENTS,
    random_state=SEED,
)
scores_flat = shared_pca.fit_transform(pooled)

diagnostics = shared_pca.get_diagnostics()
explained_variance = np.asarray(diagnostics["explained_variance_ratio_"])

print(f"Variance explained by 3 PCs : {explained_variance[:3].sum():.1%}")
print(f"Variance explained by 10 PCs: {explained_variance.sum():.1%}")
if diagnostics.get("participation_ratio_") is not None:
    print(f"Participation ratio (10-PC spectrum): {diagnostics['participation_ratio_']:.2f}")

fig_scree = plot_scree(explained_variance)
fig_scree.update_layout(title="Shared PCA: explained and cumulative variance")
fig_scree.show()

Variance explained by 3 PCs : 22.4%
Variance explained by 10 PCs: 53.9%
Participation ratio (10-PC spectrum): 9.20


## Step 5. Select Components, then Project and Baseline

The PCA scores are reshaped back to `trial × time × component`, and each trial's mean pre-stimulus score is subtracted from every time point.

<div class="alert alert-warning">
<b>⚠️ Why baseline again, in PC space?</b><br>
PCA is a rotation, and a rotation of data with a non-zero mean produces a new offset in component space. Subtracting the pre-stimulus mean <i>in PC space</i> anchors every trajectory at the origin at stimulus onset, so that "distance travelled" measures the response rather than where the participant happened to sit before the stimulus.
</div>

A shared PCA has one sign convention for the entire dataset, so no participant-specific sign alignment is needed here — unlike the per-participant fits in Step 5a.

In [7]:
scores = scores_flat.reshape(n_trials, n_times, N_COMPONENTS)
baseline_mask = times < 0
scores_baselined = scores - scores[:, baseline_mask].mean(axis=1, keepdims=True)

print(f"flat scores       : {scores_flat.shape}")
print(f"trial trajectories: {scores_baselined.shape}")
print(
    "Mean absolute baseline offset after correction: "
    f"{np.abs(scores_baselined[:, baseline_mask].mean(axis=1)).mean():.3e}"
)

flat scores       : (590352, 10)
trial trajectories: (2352, 251, 10)
Mean absolute baseline offset after correction: 1.832e-07


### Step 5a. Fit one PCA per participant

This mirrors the subject-level strategy from the EEGBCI tutorial. Each participant receives a separate PCA fitted to all three of their conditions, and we **never average the resulting PC coordinates across participants**: PC1 for one participant is not the same spatial direction as PC1 for another, because the variance ordering depends on that individual's head, sensor geometry and noise.

So why fit them at all? Because the *scalar* metrics computed later — separation, speed, path length — are invariant to sign flips and to rotations within a retained subspace. They can therefore be computed honestly in each participant's own basis, and comparing those values against the shared-basis versions tests whether a result depends on the choice of coordinate system.

The sensitivity that remains is **subspace selection**: a participant's own top three PCs may retain a different amount of their variance than the shared top three do. `METRIC_PCA_MODE` exists so that both answers can be inspected side by side.

In [8]:
subject_scores_baselined = None
subject_pcas = {}

if METRIC_PCA_MODE in {"subject", "both"}:
    subject_scores_baselined = np.empty_like(scores_baselined)

    for subject in unique_subjects:
        rows = subjects == subject
        X_subject = X[rows]
        n_subject_trials = X_subject.shape[0]
        subject_matrix = X_subject.transpose(0, 2, 1).reshape(
            n_subject_trials * n_times, n_sensors
        )

        subject_pca = DimReduction(
            method="PCA",
            n_components=N_COMPONENTS,
            random_state=SEED,
        )
        subject_flat = subject_pca.fit_transform(subject_matrix)
        subject_traj = subject_flat.reshape(
            n_subject_trials, n_times, N_COMPONENTS
        )
        subject_traj -= subject_traj[:, baseline_mask].mean(
            axis=1, keepdims=True
        )

        subject_pcas[subject] = subject_pca
        subject_scores_baselined[rows] = subject_traj

    subject_evr = pd.DataFrame({
        subject: subject_pcas[subject]
        .get_diagnostics()["explained_variance_ratio_"][:3]
        for subject in unique_subjects
    }, index=["PC1", "PC2", "PC3"]).T
    subject_evr["cumulative_3pc"] = subject_evr.sum(axis=1)
    display(subject_evr.round(3))
else:
    print("Subject-level PCA skipped: METRIC_PCA_MODE is set to shared.")

,PC1,PC2,PC3,cumulative_3pc
01,0.103,0.080,0.076,0.258
02,0.107,0.080,0.069,0.257
03,0.095,0.083,0.071,0.249


### Read the PCs before reading the geometry

Before interpreting any trajectory, look at what the axes actually are. The PC traces show *when* each axis changes, and the loading table shows which sensors contribute most strongly.

Two cautions travel with this figure. Component signs are arbitrary, so "PC2 goes up" is not a finding — timing and relative geometry are the stable quantities. And a sensor loading is not a source: a strong posterior loading is consistent with an occipital generator but does not establish one.

In [9]:
subject_condition = {}
for subject in unique_subjects:
    for condition in conditions:
        rows = (subjects == subject) & (labels == condition)
        subject_condition[subject, condition] = scores_baselined[rows].mean(axis=0)

# Channels are principal components, groups are conditions: one stacked panel
# per PC with the across-participant mean and SEM band.
pc_curves = np.stack([
    subject_condition[subject, condition][:, :N_DISPLAY_COMPONENTS].T
    for condition in conditions
    for subject in unique_subjects
])
pc_groups = [
    LABEL_NAMES[condition] for condition in conditions for _ in unique_subjects
]

fig_pc = plot_timecourses(
    pc_curves,
    times=times,
    channel_names=[f"PC{index + 1}" for index in range(N_DISPLAY_COMPONENTS)],
    group_labels=pc_groups,
    palette={LABEL_NAMES[c]: CONDITION_COLORS[c] for c in conditions},
    n_cols=1,
    error_style="band",
    xlabel="Time (s)",
    ylabel="Score (a.u.)",
    title="Principal-component time courses",
)
fig_pc.add_vline(x=0, line_dash="dash", line_color="black")
fig_pc.show()


In [10]:
loadings = np.asarray(shared_pca.get_components())[:N_DISPLAY_COMPONENTS]
loading_rows = []

for pc, weights in enumerate(loadings, start=1):
    strongest = np.argsort(np.abs(weights))[-8:][::-1]
    loading_rows.extend(
        {"component": f"PC{pc}", "sensor": channels[i], "loading": weights[i]}
        for i in strongest
    )

print("Sensors with the largest absolute loading on each displayed PC")
display(pd.DataFrame(loading_rows).round({"loading": 3}))

Sensors with the largest absolute loading on each displayed PC


,component,sensor,loading
0,PC1,MEG0433,0.247
1,PC1,MEG0732,-0.223
2,PC1,MEG2243,-0.221
3,PC1,MEG0712,0.176
4,PC1,MEG0432,-0.174
5,PC1,MEG2512,-0.167
6,PC1,MEG2432,-0.164
7,PC1,MEG0442,0.160
8,PC2,MEG2022,0.313
9,PC2,MEG2033,0.297


## Step 6. Plot Group-Mean Trajectories

This is the headline state-space visualization. Trials are averaged **within each participant and condition first**, and only then across participants, so every participant contributes equally regardless of how many trials survived rejection. The shaded 2-D envelope is the SEM across participant means.

<div class="alert alert-info">
<b>💡 How to read the geometry:</b><br>
<ul style="margin-bottom: 0;">
<li><b>Start at the origin.</b> Because of the Step 5 baseline, all conditions begin tightly clustered at (0, 0, 0) at stimulus onset. The excursion away from that cluster <i>is</i> the response.</li>
<li><b>Look for when the paths part.</b> The moment two conditions' trajectories separate is the latency at which the population representation distinguishes them — quantified in Step 7.</li>
<li><b>Rotate the 3-D plot.</b> Two trajectories can look superimposed from one angle and clearly separate from another; the 2-D projection is a shadow of a 3-D object.</li>
<li><b>A loop is a description, not a mechanism.</b> A bend or a ring in state space is a geometric fact; it does not by itself establish an oscillatory neural process.</li>
</ul>
</div>

In [11]:
group_trajectories = np.stack([
    np.stack([subject_condition[s, c] for s in unique_subjects]).mean(axis=0)
    for c in conditions
])
group_sem = np.stack([
    np.stack([subject_condition[s, c] for s in unique_subjects]).std(axis=0, ddof=1)
    / np.sqrt(len(unique_subjects))
    for c in conditions
])

plot_labels = np.array([LABEL_NAMES[c] for c in conditions])
color_map = {LABEL_NAMES[c]: CONDITION_COLORS[c] for c in conditions}

fig_2d = plot_trajectory(
    X=group_trajectories[..., :2],
    times=times,
    labels=plot_labels,
    color_map=color_map,
    title="Equal-participant mean trajectories: PC1–PC2",
    dimensions=2,
    smooth_window=12,
    show_markers=False,
    add_start_end_markers=True,
)
fig_2d.show()

In [12]:
fig_3d = plot_trajectory(
    X=group_trajectories[..., :3],
    times=times,
    labels=plot_labels,
    color_map=color_map,
    smooth_window=12,
    show_markers=False,
    linewidth=10,                      # Auto-scales all trace line widths
    title="Equal-participant mean trajectories: PC1–PC2–PC3",
    dimensions=3,
    add_start_end_markers=True,
    height=700,
)
fig_3d.show()

## Step 7. Compare Conditions

The two planned contrasts are evaluated **per participant**, then summarized:

- **Famous vs Unfamiliar** — the Euclidean distance between those two condition-mean trajectories.
- **Faces vs Scrambled** — Famous and Unfamiliar are first averaged with equal weight, then the distance to Scrambled is measured.

Each participant's mean pre-stimulus distance is subtracted, so the curves start at zero and any lift is a response rather than a baseline offset.

`METRIC_PCA_MODE` determines whether these distances are computed in the shared basis, each participant's own basis, or both.

<div class="alert alert-success">
<b>✅ What a convincing effect looks like:</b><br>
It emerges <i>after</i> stimulus onset, it appears in more than one participant rather than only in the thick group-average line, and its timing is stable across the reasonable analysis choices tested in Steps 10 and 11.
</div>

<div class="alert alert-warning">
<b>⚠️ Read the onset, not only the argmax:</b><br>
For the face/scrambled contrast the informative feature is the <b>sharp lift beginning around 120–170 ms</b> — the N170 range, recovered by an analysis that was never told faces exist. The <i>maximum</i> of the curve typically sits considerably later, on a broad sustained plateau where the exact argmax is close to arbitrary. Reporting that maximum as "the latency of the face effect" would badly misdescribe the timecourse. Familiarity, by contrast, has no early component at all and rises only in the second half of the epoch.
</div>

In [13]:
def compute_contrast_curves(trajectories, trial_labels):
    # Return baseline-relative separation curves for each participant.
    curves = {"Faces vs Scrambled": [], "Famous vs Unfamiliar": []}
    for subject in unique_subjects:
        means = {
            condition: trajectories[
                (subjects == subject) & (trial_labels == condition)
            ].mean(axis=0)
            for condition in conditions
        }
        faces = 0.5 * (means[1] + means[2])
        distances = {
            "Faces vs Scrambled": np.linalg.norm(faces - means[3], axis=1),
            "Famous vs Unfamiliar": np.linalg.norm(
                means[1] - means[2], axis=1
            ),
        }
        for name, distance in distances.items():
            curves[name].append(distance - distance[baseline_mask].mean())
    return {name: np.asarray(values) for name, values in curves.items()}


metric_trajectory_spaces = {}
if METRIC_PCA_MODE in {"shared", "both"}:
    metric_trajectory_spaces["Shared PCA"] = scores_baselined[
        ..., :N_DISPLAY_COMPONENTS
    ]
if METRIC_PCA_MODE in {"subject", "both"}:
    metric_trajectory_spaces["Subject PCA"] = subject_scores_baselined[
        ..., :N_DISPLAY_COMPONENTS
    ]

contrast_curves_by_space = {
    space: compute_contrast_curves(trajectories, labels)
    for space, trajectories in metric_trajectory_spaces.items()
}

In [14]:
contrast_names = ["Faces vs Scrambled", "Famous vs Unfamiliar"]

# Each panel layers two coco-pipe figures: the per-participant curves
# underneath, the group mean with its SEM band on top.
contrast_panels = {}
for space, space_curves in contrast_curves_by_space.items():
    for name in contrast_names:
        curves = np.asarray(space_curves[name])
        color = CONTRAST_COLORS[name]
        participants = plot_trajectory_metric_series(
            curves,
            times=times,
            labels=np.array([f"sub-{subject}" for subject in unique_subjects]),
            color_map={f"sub-{subject}": color for subject in unique_subjects},
            title=f"{space}: {name}",
            ylabel="Baseline-relative distance",
        )
        participants.update_traces(line={"width": 1}, showlegend=False)
        group = plot_timecourses(
            curves[:, np.newaxis, :],
            times=times,
            channel_names=[name],
            group_labels=[name] * len(curves),
            palette={name: color},
            error_style="band",
            xlabel="Time (s)",
            ylabel="Baseline-relative distance",
            title=f"{space}: {name}",
        )
        contrast_panels[f"{space}: {name}"] = overlay_figures(
            [participants, group], opacities=[0.28, None]
        )

fig_contrasts = facet_figures(
    contrast_panels,
    n_cols=2,
    title="Planned contrasts: participants and group mean",
    row_height=380,
)
fig_contrasts.show()


In [15]:
active_mask = (times >= ACTIVE_WINDOW[0]) & (times <= ACTIVE_WINDOW[1])
active_times = times[active_mask]
summary_rows = []

for space, space_curves in contrast_curves_by_space.items():
    for name, curves in space_curves.items():
        for subject, curve in zip(unique_subjects, curves):
            active_curve = curve[active_mask]
            peak_index = int(np.argmax(active_curve))
            summary_rows.append({
                "pca_space": space,
                "subject": subject,
                "contrast": name,
                "auc_0_600ms": np.trapezoid(active_curve, active_times),
                "peak_separation": active_curve[peak_index],
                "peak_time_s": active_times[peak_index],
            })

contrast_summary = pd.DataFrame(summary_rows)
display(contrast_summary.round(3))
display(
    contrast_summary.groupby(["pca_space", "contrast"])
    .agg(
        mean_auc=("auc_0_600ms", "mean"),
        sem_auc=("auc_0_600ms", "sem"),
        mean_peak_time_s=("peak_time_s", "mean"),
    )
    .round(3)
)

,pca_space,subject,contrast,auc_0_600ms,peak_separation,peak_time_s
0,Shared PCA,01,Faces vs Scrambled,2.404,8.225,0.244
1,Shared PCA,02,Faces vs Scrambled,1.207,4.019,0.272
2,Shared PCA,03,Faces vs Scrambled,1.393,4.855,0.492
3,Shared PCA,01,Famous vs Unfamiliar,0.201,2.220,0.308
4,Shared PCA,02,Famous vs Unfamiliar,0.879,3.215,0.552
5,Shared PCA,03,Famous vs Unfamiliar,0.509,2.767,0.400
6,Subject PCA,01,Faces vs Scrambled,2.879,8.919,0.244
7,Subject PCA,02,Faces vs Scrambled,1.135,3.676,0.420
8,Subject PCA,03,Faces vs Scrambled,1.506,5.212,0.496
9,Subject PCA,01,Famous vs Unfamiliar,0.203,1.969,0.148


mean_auc  sem_auc  mean_peak_time_s
pca_space   contrast                                                 
Shared PCA  Faces vs Scrambled       1.668    0.372             0.336
            Famous vs Unfamiliar     0.530    0.196             0.420
Subject PCA Faces vs Scrambled       1.840    0.530             0.387
            Famous vs Unfamiliar     0.562    0.186             0.365

## Step 8. Compare the Planned Effects with a Within-Participant Null

A separation curve that rises after stimulus onset looks convincing, but distances between noisy means are positive by construction — the question is whether *this much* separation is more than label noise would produce.

Because the PCA was fitted without labels, its axes stay fixed under a label shuffle. We therefore shuffle condition labels **within each participant**, preserving that participant's trial counts and preprocessing history, and recompute both planned contrasts on every shuffle. Shuffling within participant is what keeps the null honest: a global shuffle would also destroy the participant structure and produce an artificially easy null to beat.

<div class="alert alert-warning">
<b>⚠️ The p-value floor:</b><br>
Two hundred permutations keep this tutorial practical, but they limit the smallest attainable empirical p-value to 1/201 ≈ 0.005. If the observed value exceeds every permutation, report <b>p &lt; 0.005</b> — the floor — rather than the literal printed number, which is not an estimate of how small the true value is. A final analysis should use more permutations and more participants.
</div>

Read the histogram by locating the observed AUC relative to the shuffled distribution.

In [16]:
N_PERMUTATIONS = 200
observed_auc = {}
null_auc = {}

for space, trajectories in metric_trajectory_spaces.items():
    observed_auc[space] = {
        name: np.trapezoid(curves.mean(axis=0)[active_mask], active_times)
        for name, curves in contrast_curves_by_space[space].items()
    }
    null_auc[space] = {
        name: [] for name in contrast_curves_by_space[space]
    }

for _ in range(N_PERMUTATIONS):
    shuffled = labels.copy()
    for subject in unique_subjects:
        rows = np.flatnonzero(subjects == subject)
        shuffled[rows] = rng.permutation(shuffled[rows])

    for space, trajectories in metric_trajectory_spaces.items():
        permuted_curves = compute_contrast_curves(trajectories, shuffled)
        for name, curves in permuted_curves.items():
            null_auc[space][name].append(
                np.trapezoid(
                    curves.mean(axis=0)[active_mask], active_times
                )
            )

inference_rows = []
for space, space_nulls in null_auc.items():
    for name, values in space_nulls.items():
        values = np.asarray(values)
        null_auc[space][name] = values
        inference_rows.append({
            "pca_space": space,
            "contrast": name,
            "observed_auc": observed_auc[space][name],
            "null_mean": values.mean(),
            "empirical_p": (
                1 + np.sum(values >= observed_auc[space][name])
            ) / (N_PERMUTATIONS + 1),
        })

inference = pd.DataFrame(inference_rows)
display(inference.round(4))

,pca_space,contrast,observed_auc,null_mean,empirical_p
0,Shared PCA,Faces vs Scrambled,1.6677,0.1619,0.005
1,Shared PCA,Famous vs Unfamiliar,0.5299,0.1916,0.005
2,Subject PCA,Faces vs Scrambled,1.8404,0.1842,0.005
3,Subject PCA,Famous vs Unfamiliar,0.5621,0.2134,0.005


In [17]:
fig_null = make_subplots(
    rows=len(null_auc),
    cols=2,
    subplot_titles=[
        f"{space}: {contrast}"
        for space in null_auc
        for contrast in contrast_names
    ],
)

for row, (space, space_nulls) in enumerate(null_auc.items(), start=1):
    for column, name in enumerate(contrast_names, start=1):
        values = space_nulls[name]
        color = CONTRAST_COLORS[name]
        fig_null.add_trace(
            go.Histogram(
                x=values,
                nbinsx=25,
                marker_color=color,
                opacity=0.72,
                showlegend=False,
            ),
            row=row,
            col=column,
        )
        fig_null.add_vline(
            x=observed_auc[space][name],
            line_color="black",
            line_width=3,
            annotation_text="observed",
            row=row,
            col=column,
        )

fig_null.update_xaxes(title_text="Baseline-relative separation AUC")
fig_null.update_yaxes(title_text="Permutations", col=1)
fig_null.update_layout(
    height=360 * len(null_auc),
    title="Within-participant permutation nulls",
)
fig_null.show()

## Step 9. Trajectory Metrics & Dynamical Systems

Speed is the rate of movement through PCA space: how quickly the population state is changing, independently of *where* it is. We compute it from each participant's condition-mean trajectory and then summarize across participants.

<div class="alert alert-success">
<b>🧠 A null here is informative, not a failure:</b><br>
In many neural datasets the largest dynamical component is <b>condition-invariant</b> — a large, stereotyped state transition that occurs on every trial regardless of what was shown, occupying dimensions orthogonal to the tuned ones. If the speed timecourses show a common post-stimulus profile with no condition difference, that is the expected signature of such a component, not an absent result. It says the conditions differ in <i>where</i> the trajectory goes rather than in <i>how fast</i> it gets there.
</div>

Numerical derivatives amplify noise, so interpret broad changes in the speed profile rather than isolated samples.

In [18]:
# One panel per PCA space: participant speed curves under the condition means
# and their SEM bands.
speed_panels = {}
for space, trajectories in metric_trajectory_spaces.items():
    curves, groups = [], []
    for condition in conditions:
        for subject in unique_subjects:
            subject_mean = trajectories[
                (subjects == subject) & (labels == condition)
            ].mean(axis=0)
            velocity = np.gradient(subject_mean, times, axis=0)
            curves.append(np.linalg.norm(velocity, axis=1))
            groups.append(LABEL_NAMES[condition])

    stacked = np.asarray(curves)
    participants = plot_trajectory_metric_series(
        stacked,
        times=times,
        labels=np.array(groups),
        color_map={LABEL_NAMES[c]: CONDITION_COLORS[c] for c in conditions},
        title=space,
        ylabel="Speed (a.u./s)",
    )
    participants.update_traces(line={"width": 1}, showlegend=False)
    group = plot_timecourses(
        stacked[:, np.newaxis, :],
        times=times,
        channel_names=["speed"],
        group_labels=groups,
        palette={LABEL_NAMES[c]: CONDITION_COLORS[c] for c in conditions},
        error_style="band",
        xlabel="Time (s)",
        ylabel="Speed (a.u./s)",
        title=space,
    )
    speed_panels[space] = overlay_figures([participants, group], opacities=[0.25, None])

fig_speed = facet_figures(
    speed_panels,
    n_cols=1,
    title="Trajectory speed: participants and condition means",
    row_height=380,
    shared_yaxes=False,
)
fig_speed.show()


## Step 10. Validate — Is the Shared Three-PC Result Fragile?

Every result so far rests on a choice of three components. This step asks whether it survives that choice being made differently.

<div class="alert alert-warning">
<b>⚠️ Compare shapes, not magnitudes:</b><br>
Distances almost always grow when orthogonal dimensions are added, simply because there is more room to be far apart in. Raw AUC values computed with 2, 3 and 5 PCs are therefore <b>not</b> directly comparable. The stable, comparable quantities are the <b>shape of the timecourse</b> and the <b>peak latency</b> — if those hold across component counts, the effect is not an artifact of the display dimension.
</div>

In [19]:
reference_curves = {
    name: curves.mean(axis=0)
    for name, curves in compute_contrast_curves(
        scores_baselined[..., :3], labels
    ).items()
}
sensitivity_rows = []

for n_components in (2, 3, 5):
    candidate = compute_contrast_curves(
        scores_baselined[..., :n_components], labels
    )
    for name, curves in candidate.items():
        mean_curve = curves.mean(axis=0)
        active_curve = mean_curve[active_mask]
        sensitivity_rows.append({
            "contrast": name,
            "components": n_components,
            "r_with_3pc_timecourse": np.corrcoef(
                active_curve, reference_curves[name][active_mask]
            )[0, 1],
            "peak_time_s": active_times[int(np.argmax(active_curve))],
        })

sensitivity = pd.DataFrame(sensitivity_rows)
display(sensitivity.round(3))

,contrast,components,r_with_3pc_timecourse,peak_time_s
0,Faces vs Scrambled,2,0.974,0.320
1,Famous vs Unfamiliar,2,0.756,0.300
2,Faces vs Scrambled,3,1.000,0.320
3,Famous vs Unfamiliar,3,1.000,0.308
4,Faces vs Scrambled,5,0.972,0.320
5,Famous vs Unfamiliar,5,0.971,0.308


## Step 11. Fit Focused PCA Spaces for Two-Condition Questions

The three-condition PCA remains the primary analysis, because every condition shares one coordinate system and the contrasts are therefore commensurable. But a shared fit allocates its axes to whatever varies most overall, which may not be what distinguishes a particular pair.

So we fit two additional shared PCAs, each across all selected participants but only the trials of one pair:

1. **Famous and Unfamiliar only**
2. **Famous and Scrambled only**

<div class="alert alert-warning">
<b>⚠️ Do not compare axes across these models:</b><br>
PC1 in one focused PCA is not PC1 in another, and neither is PC1 of the three-condition fit. Compare the <i>timing and consistency</i> of divergence across models — never raw coordinates, and never raw distances between different models.
</div>

In [20]:
FOCUSED_PAIRS = {
    "Famous vs Unfamiliar": (1, 2),
    "Famous vs Scrambled": (1, 3),
}
focused_results = {}

for pair_name, pair in FOCUSED_PAIRS.items():
    pair_mask = np.isin(labels, pair)
    X_pair = X[pair_mask]
    labels_pair = labels[pair_mask]
    subjects_pair = subjects[pair_mask]
    n_pair_trials = X_pair.shape[0]
    pair_matrix = X_pair.transpose(0, 2, 1).reshape(
        n_pair_trials * n_times, n_sensors
    )

    pair_pca = DimReduction(
        method="PCA",
        n_components=N_COMPONENTS,
        random_state=SEED,
    )
    pair_flat = pair_pca.fit_transform(pair_matrix)
    pair_scores = pair_flat.reshape(
        n_pair_trials, n_times, N_COMPONENTS
    )
    pair_scores -= pair_scores[:, baseline_mask].mean(
        axis=1, keepdims=True
    )

    subject_pair_means = {
        (subject, condition): pair_scores[
            (subjects_pair == subject) & (labels_pair == condition)
        ].mean(axis=0)
        for subject in unique_subjects
        for condition in pair
    }
    pair_group = np.stack([
        np.stack([
            subject_pair_means[subject, condition]
            for subject in unique_subjects
        ]).mean(axis=0)
        for condition in pair
    ])
    pair_sem = np.stack([
        np.stack([
            subject_pair_means[subject, condition]
            for subject in unique_subjects
        ]).std(axis=0, ddof=1) / np.sqrt(len(unique_subjects))
        for condition in pair
    ])
    separation = np.stack([
        np.linalg.norm(
            subject_pair_means[subject, pair[0]][:, :3]
            - subject_pair_means[subject, pair[1]][:, :3],
            axis=1,
        )
        for subject in unique_subjects
    ])
    separation -= separation[:, baseline_mask].mean(
        axis=1, keepdims=True
    )

    focused_results[pair_name] = {
        "pair": pair,
        "pca": pair_pca,
        "scores": pair_scores,
        "group": pair_group,
        "sem": pair_sem,
        "separation": separation,
        "variance_3pc": pair_pca
        .get_diagnostics()["explained_variance_ratio_"][:3].sum(),
    }

focused_variance = pd.DataFrame([
    {
        "focused_space": pair_name,
        "variance_explained_3pc": result["variance_3pc"],
    }
    for pair_name, result in focused_results.items()
])
display(focused_variance.round(3))

focused_summary_rows = []
for pair_name, result in focused_results.items():
    for subject, curve in zip(unique_subjects, result["separation"]):
        active_curve = curve[active_mask]
        peak_index = int(np.argmax(active_curve))
        focused_summary_rows.append({
            "focused_space": pair_name,
            "subject": subject,
            "auc_0_600ms": np.trapezoid(active_curve, active_times),
            "peak_separation": active_curve[peak_index],
            "peak_time_s": active_times[peak_index],
        })

focused_summary = pd.DataFrame(focused_summary_rows)
display(focused_summary.round(3))
display(
    focused_summary.groupby("focused_space")
    .agg(
        mean_auc=("auc_0_600ms", "mean"),
        sem_auc=("auc_0_600ms", "sem"),
        mean_peak_time_s=("peak_time_s", "mean"),
    )
    .round(3)
)

,focused_space,variance_explained_3pc
0,Famous vs Unfamiliar,0.225
1,Famous vs Scrambled,0.224


,focused_space,subject,auc_0_600ms,peak_separation,peak_time_s
0,Famous vs Unfamiliar,01,0.204,2.285,0.308
1,Famous vs Unfamiliar,02,0.895,3.305,0.552
2,Famous vs Unfamiliar,03,0.527,2.809,0.400
3,Famous vs Scrambled,01,2.573,8.599,0.316
4,Famous vs Scrambled,02,1.460,4.505,0.272
5,Famous vs Scrambled,03,1.059,3.973,0.492


,mean_auc,sem_auc,mean_peak_time_s
focused_space,,,
Famous vs Scrambled,1.697,0.453,0.36
Famous vs Unfamiliar,0.542,0.200,0.42


In [21]:
for pair_name, result in focused_results.items():
    pair = result["pair"]
    pair_labels = np.array([LABEL_NAMES[c] for c in pair])
    pair_colors = {LABEL_NAMES[c]: CONDITION_COLORS[c] for c in pair}

    figure = plot_trajectory(
        X=result["group"][..., :3],
        times=times,
        labels=pair_labels,
        color_map=pair_colors,
        title=f"Focused shared PCA: {pair_name}",
        dimensions=3,
        show_markers=False,
        smooth_window=12,
        add_start_end_markers=True,
    )
    figure.show()

In [22]:
# Same two-layer construction as the planned contrasts, per focused pair.
focused_panels = {}
for pair_name, result in focused_results.items():
    curves = np.asarray(result["separation"])
    color = CONTRAST_COLORS.get(pair_name, "#444444")
    participants = plot_trajectory_metric_series(
        curves,
        times=times,
        labels=np.array([f"sub-{subject}" for subject in unique_subjects]),
        color_map={f"sub-{subject}": color for subject in unique_subjects},
        title=pair_name,
        ylabel="Baseline-relative distance",
    )
    participants.update_traces(line={"width": 1}, showlegend=False)
    group = plot_timecourses(
        curves[:, np.newaxis, :],
        times=times,
        channel_names=[pair_name],
        group_labels=[pair_name] * len(curves),
        palette={pair_name: color},
        error_style="band",
        xlabel="Time (s)",
        ylabel="Baseline-relative distance",
        title=pair_name,
    )
    focused_panels[pair_name] = overlay_figures(
        [participants, group], opacities=[0.28, None]
    )

fig_focused_sep = facet_figures(
    focused_panels,
    n_cols=2,
    title="Focused-PCA separation: participants and group mean",
    row_height=380,
)
fig_focused_sep.show()


### Interpret the focused spaces

Use these figures to ask a single question: does a pair become easier to see when PCA is allowed to focus on it?

- **Agreement in divergence timing** with the three-condition space strengthens the result — the effect is robust to how the axes were chosen.
- **A dramatic change that appears only in a focused PCA** suggests the effect is weak relative to the variance contributed by the excluded third condition. That is worth reporting, but it is a weaker claim than one that survives in the common space.

## Conclusions & Interpretation Checklist

The notebook has now answered the same questions in three complementary coordinate systems:

- **One three-condition shared PCA** provides the common group trajectory and the primary contrasts.
- **Participant-specific PCAs** test whether separation and speed depend on using a shared basis.
- **Two focused shared PCAs** ask whether Famous–Unfamiliar or Famous–Scrambled structure is clearer when the third condition is excluded.

Before drawing a conclusion, check the chain of evidence:

- **The PCA space and the statistical unit are separate choices.** Group figures require shared axes; metrics are always computed per participant *before* group summarisation.
- **Labels never entered the fit,** so a separation found in Step 7 was present in the unlabelled variance structure.
- **The null was built within participant,** preserving trial counts and preprocessing history.
- **The peak latency survived** the component-count sensitivity check (Step 10) and the focused-PCA check (Step 11).
- **Sensor-space geometry does not establish anatomical generators** or a mechanistic dynamical model, and a loop in state space is a description rather than an oscillator.
- **Sample size.** These numerical results are demonstrations on a modest cohort, not population estimates.

<div class="alert alert-success">
<b>🎯 Main takeaway:</b><br>
An analysis that was never told faces exist recovers a face-versus-scrambled divergence in the N170 range, and places familiarity several hundred milliseconds later. The value of the trajectory framing is not that it beats an ERF analysis at detecting these effects — it is that latency, geometry and dynamics come out of one coherent description of the population state.
</div>

<div class="alert alert-info">
<b>➡️ Next representation: band-limited envelopes.</b><br>
Broadband ERF trajectories emphasize phase-locked responses, which is only part of what MEG measures. The spectral companion tutorial applies MNE filtering followed by the Hilbert amplitude envelope, then runs this same PCA logic on alpha, beta and gamma activity — where the interesting effects are induced rather than evoked and would largely cancel in the average used here.
</div>

## Running the Same Analysis Headlessly

The companion script repeats every computation in this notebook, and additionally writes the full output bundle and a self-contained HTML report:

```bash
python scripts/analysis_megfaces_main.py \
    --derivatives-root <prepared-derivatives> \
    --output outputs/megfaces_main
```

Useful variations:

```bash
# quick validation run on three participants
python scripts/analysis_megfaces_main.py --smoke

# more permutations for the Step 8 null
python scripts/analysis_megfaces_main.py --n-perm 1000

# restrict to one helmet sensor selection
python scripts/analysis_megfaces_main.py --sensor-set sensors_occipital
```